# CasCrop: Polarity-Routed Cascade Network (PRCN)
**Full experiments for Nature Communications submission.**

Three novel components:
1. **Polarity-Routed Diffusion** — negative shocks → commodity graph, positive → geographic graph
2. **Cascade Decay Signatures** — hop-by-hop attenuation encodes systemic vs local risk
3. **Vulnerability-Conditioned Routing** — biophysical state selects active contagion channels

5-row ablation × 5 seeds × 100 epochs on 638K samples. ~30 min on T4 GPU.

In [ ]:
#@title Configuration
QUICK_TEST = True  #@param {type:"boolean"}
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 30 if QUICK_TEST else 100
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 2048 if QUICK_TEST else 1024
print(f"{'QUICK' if QUICK_TEST else 'FULL'}: {len(SEEDS)} seeds, {EPOCHS} epochs")

In [ ]:
#@title Setup (Colab + Kaggle compatible)
import torch, os, sys, json, time, shutil, subprocess
import numpy as np, pandas as pd
from pathlib import Path

# Detect platform
ON_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
ON_KAGGLE = os.path.exists('/kaggle/working')
WORKDIR = '/kaggle/working' if ON_KAGGLE else '/content'
print(f'Platform: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f}GB)')
    if gpu_mem < 8: BATCH_SIZE = min(BATCH_SIZE, 512)
else:
    print('NO GPU — enable GPU in runtime settings')

if not os.path.exists('CasCrop'):
    !git clone https://github.com/keshavkrishnan08/CasCrop.git
if os.path.basename(os.getcwd()) != 'CasCrop':
    os.chdir('CasCrop')
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn tqdm pyyaml 2>&1 | tail -1
sys.path.insert(0, 'src')
for d in ['checkpoints','results','paper/figures','paper/tables']:
    os.makedirs(d, exist_ok=True)

SAVE_TO_DRIVE = False; DRIVE_PATH = ''
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'
        os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
        SAVE_TO_DRIVE = True; print(f'Drive: {DRIVE_PATH}')
    except: print('No Drive')

def backup():
    if not SAVE_TO_DRIVE: return
    for d in ['results','checkpoints','paper/figures','paper/tables']:
        if not os.path.exists(d): continue
        dst = f'{DRIVE_PATH}/{d}'; os.makedirs(dst, exist_ok=True)
        for f in Path(d).glob('*'):
            if f.is_file(): shutil.copy2(f, f'{dst}/{f.name}')
print('OK')

In [ ]:
#@title Pre-compute Polarity-Routed Cascade Features (~15 sec)
if Path('data/processed/features_monthly.parquet').exists() and Path('data/graphs/combined_graph.npz').exists():
    print('Raw data present.')
else:
    !wget -q --show-progress -O monthly.tar.gz https://github.com/keshavkrishnan08/CasCrop/releases/download/v0.1-data/cascrop_monthly.tar.gz
    !tar xzf monthly.tar.gz && rm monthly.tar.gz

assert Path('data/processed/features_monthly.parquet').exists(), 'Data missing!'
assert Path('data/graphs/combined_graph.npz').exists(), 'Graph missing!'

# Always recompute to pick up latest feature set
print('Pre-computing polarity-routed cascade features...')
!rm -f data/processed/features_cascade.parquet
!python scripts/precompute_cascade.py

assert Path('data/processed/features_cascade.parquet').exists(), 'Cascade pre-computation failed!'

f = pd.read_parquet('data/processed/features_cascade.parquet')
l = pd.read_parquet('data/processed/labels_monthly.parquet')
with open('data/processed/feature_groups_cascade.json') as fj: groups = json.load(fj)
print(f'{len(f):,} samples | {f["fips"].nunique()} counties | {l["waste"].mean():.1%} waste')
print(f'Features: bio={len(groups["biophysical"])}, econ={len(groups["economic"])}, hist={len(groups["historical"])}, cascade={len(groups["cascade"])}')
print(f'Cascade channels: {groups["cascade"]}')

In [ ]:
#@title Smoke Test
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
from models.prcn import PRCN
_N = 64
_batch = {
    'x_bio': torch.randn(_N, len(groups['biophysical']), device=device),
    'x_econ': torch.randn(_N, len(groups['economic']), device=device),
    'x_hist': torch.randn(_N, len(groups['historical']), device=device),
    'x_cascade': torch.randn(_N, len(groups['cascade']), device=device),
}
model = PRCN(
    bio_dim=len(groups['biophysical']), econ_dim=len(groups['economic']),
    hist_dim=len(groups['historical']), cascade_dim=len(groups['cascade']),
).to(device)
out = model(_batch)
out['waste_logits'].sum().backward()
print(f'PRCN: {sum(p.numel() for p in model.parameters()):,} params on {device}')
print(f'Channel routing weights shape: {out["channel_weights"].shape}')
print(f'Persistence score shape: {out["persistence_score"].shape}')
del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Smoke test passed')

---
## Experiment 1: 6-Row Ablation
| Row | Model | Novel Component Added |
|-----|-------|----------------------|
| R1 | Local Only | Biophysical + historical (no graph) |
| R2 | Local + Econ | + Economic features |
| R3 | Symmetric Diff | + Graph diffusion (same graph, collapse +/-) |
| R4 | Polarity-Routed | + **Novel:** neg→commodity graph, pos→geo graph |
| R5 | PRCN | + **Novel:** cascade decay + vulnerability routing + cross-commodity |
| R6 | **Temporal PRCN** | + **Novel:** GRU contagion momentum over 6-month windows |

In [ ]:
%%time
models_str = 'local_only local_econ cascade_direct symmetric_diff polarity_routed prcn'
ss = ' '.join(str(s) for s in SEEDS)

print(f'Training 6 ablation models x {len(SEEDS)} seeds x {EPOCHS} epochs...')
r = subprocess.run(
    f'python scripts/train_prcn.py --models {models_str} --seeds {ss} '
    f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0',
    shell=True, capture_output=True, text=True, timeout=36000)
for line in r.stdout.strip().split('\n')[-30:]: print(line)
if r.returncode != 0:
    print(f'EXIT CODE: {r.returncode}')
    if r.stderr: print(f'STDERR:\n{r.stderr[-1000:]}')
backup()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Results
models = ['local_only', 'local_econ', 'symmetric_diff', 'polarity_routed', 'prcn', 'temporal_prcn']
if os.path.exists('results/prcn_results.json'):
    with open('results/prcn_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    print(f'{"Model":<20} {"AUC-ROC":>14} {"F1":>8} {"AP":>8}')
    print('-'*55)
    for m in models:
        d = df[(df['model']==m) & (df['test_auc_roc']>0)]
        if len(d):
            print(f'{m:<20} {d["test_auc_roc"].mean():.3f}+/-{d["test_auc_roc"].std():.3f}'
                  f'  {d["test_f1"].mean():.3f}  {d["test_auc_pr"].mean():.3f}')
    best = 'temporal_prcn' if 'temporal_prcn' in df['model'].values else 'prcn'
    print()
    for base in [m for m in models if m != best]:
        c = df[df['model']==best]['test_auc_roc'].mean()
        b = df[df['model']==base]['test_auc_roc'].mean()
        print(f'{best} vs {base:<18} DAUC={c-b:+.4f}')
else:
    print('No results yet')

---
## Statistical Tests

In [ ]:
if os.path.exists('results/prcn_results.json'):
    with open('results/prcn_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    from evaluation.statistical_tests import paired_ttest_across_seeds
    best = 'temporal_prcn' if 'temporal_prcn' in df['model'].values else 'prcn'
    ca = sorted(df[df['model']==best]['test_auc_roc'].tolist())
    if len(ca) >= 2:
        print(f'{"Comparison":<45} {"DAUC":>7} {"p":>8} {"Sig":>5}')
        print('-'*68)
        for m in ['local_only','local_econ','symmetric_diff','polarity_routed','prcn','temporal_prcn']:
            if m == best: continue
            ma = sorted(df[df['model']==m]['test_auc_roc'].tolist())
            if len(ma) != len(ca): continue
            t = paired_ttest_across_seeds(ca, ma)
            sig = '***' if t['p_value']<.001 else '**' if t['p_value']<.01 else '*' if t['p_value']<.05 else 'n.s.'
            print(f'{best} vs {m:<30} {t["mean_diff"]:>+.4f} {t["p_value"]:>8.4f} {sig:>5}')
else:
    print('No results yet')

---
## Figure: 5-Row Ablation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size':9,'figure.dpi':300})

if 'df' in dir() and len(df) > 0:
    mo = ['local_only','local_econ','symmetric_diff','polarity_routed','prcn','temporal_prcn']
    dn = ['R1: Local\n(Bio only)','R2: +Econ\n(No graph)','R3: +Symm\nDiffusion',
          'R4: +Polarity\nRouting','R5: PRCN\n(Snapshot)','R6: T-PRCN\n(Momentum)']
    co = ['#95a5a6','#3498db','#9b59b6','#e67e22','#e74c3c','#c0392b']
    avail = [m for m in mo if m in df['model'].values]
    ms = [df[df['model']==m]['test_auc_roc'].mean() for m in avail]
    ss_ = [df[df['model']==m]['test_auc_roc'].std() if len(df[df['model']==m])>1 else 0 for m in avail]
    labels = [dn[mo.index(m)] for m in avail]
    colors = [co[mo.index(m)] for m in avail]
    
    fig, ax = plt.subplots(figsize=(9,3.5))
    bars = ax.bar(range(len(avail)), ms, 0.6, yerr=ss_, capsize=4, color=colors, edgecolor='k', linewidth=.5)
    for b, v in zip(bars, ms):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.3f}', ha='center', fontsize=7)
    ax.set_xticks(range(len(avail))); ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylim(0.7, 1.0); ax.set_ylabel('AUC-ROC'); ax.grid(axis='y', alpha=0.3)
    ax.set_title('PRCN Ablation: Each Row Adds One Novel Component', fontweight='bold')
    plt.tight_layout()
    fig.savefig('paper/figures/fig3_prcn_ablation.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig3_prcn_ablation.pdf')

---
## Experiment 2: Graph Perturbation

In [ ]:
%%time
import shutil

# Back up
for src_f in ['data/graphs/combined_graph.npz', 'data/processed/features_cascade.parquet']:
    shutil.copy(src_f, src_f + '.bak')
if os.path.exists('results/prcn_results.json'):
    shutil.copy('results/prcn_results.json', 'results/prcn_results_main.json')
for f in Path('checkpoints').glob('prcn_prcn_*.pt'):
    shutil.copy(f, f'{f}.bak')

# Shuffle graph edges
g = np.load('data/graphs/combined_graph.npz.bak'); np.random.seed(42)
np.savez('data/graphs/combined_graph.npz',
         edge_index=np.array([g['edge_index'][0], np.random.permutation(g['edge_index'][1])]),
         edge_weight=g['edge_weight'])

# Re-compute cascade on shuffled graph
os.remove('data/processed/features_cascade.parquet')
!python scripts/precompute_cascade.py 2>&1 | tail -3

# Train PRCN on shuffled
ss3 = ' '.join(str(s) for s in SEEDS[:3])
r = subprocess.run(
    f'python scripts/train_prcn.py --models prcn --seeds {ss3} '
    f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0',
    shell=True, capture_output=True, text=True, timeout=3600)
for line in r.stdout.strip().split('\n')[-10:]: print(line)

# Save perturbation results
if os.path.exists('results/prcn_results.json'):
    shutil.copy('results/prcn_results.json', 'results/perturbation_results.json')

# Restore
shutil.copy('data/graphs/combined_graph.npz.bak', 'data/graphs/combined_graph.npz')
shutil.copy('data/processed/features_cascade.parquet.bak', 'data/processed/features_cascade.parquet')
for f in Path('checkpoints').glob('prcn_prcn_*.pt.bak'):
    shutil.copy(f, str(f).replace('.bak', '')); f.unlink()
if os.path.exists('results/prcn_results_main.json'):
    shutil.copy('results/prcn_results_main.json', 'results/prcn_results.json')

# Compare
if os.path.exists('results/perturbation_results.json'):
    with open('results/perturbation_results.json') as fp: pert = json.load(fp)
    with open('results/prcn_results.json') as fm: main = json.load(fm)
    pert_auc = pd.DataFrame(pert).query('model=="prcn"')['test_auc_roc'].mean()
    main_auc = pd.DataFrame(main).query('model=="prcn"')['test_auc_roc'].mean()
    print(f'\nPerturbation Analysis:')
    print(f'  Real graph AUC:     {main_auc:.4f}')
    print(f'  Shuffled graph AUC: {pert_auc:.4f}')
    print(f'  Delta:              {main_auc - pert_auc:+.4f}')
    print(f'  Graph signal: {"CONFIRMED" if main_auc > pert_auc else "NOT CONFIRMED"}')

if torch.cuda.is_available(): torch.cuda.empty_cache()

---
## Case Study: Contagion Channel Activation During 2023 Drought

In [ ]:
#@title Case Study: Channel Weights for High-Loss County in 2023
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size':9,'figure.dpi':300})

# Find worst county in 2023 (test set)
feat = pd.read_parquet('data/processed/features_cascade.parquet')
labs = pd.read_parquet('data/processed/labels_monthly.parquet')
with open('data/processed/feature_groups_cascade.json') as fj: grp = json.load(fj)
with open('data/processed/stats_monthly.json') as fj: stats = json.load(fj)

county_waste = labs[labs['year']==2023].groupby('fips')['waste'].mean().sort_values(ascending=False)
target_fips = county_waste.index[0]
target_comm = feat[(feat['fips']==target_fips) & (feat['year']==2023)]['commodity'].mode().iloc[0]
print(f'Case study: FIPS {target_fips}, {target_comm}, waste rate={county_waste.iloc[0]:.0%}')

# Get monthly data for this county-commodity in 2023
mask = (feat['fips']==target_fips) & (feat['commodity']==target_comm) & (feat['year']==2023)
county_data = feat[mask].sort_values('month')

# Load trained PRCN and run inference
from models.prcn import PRCN
ckpt_path = 'checkpoints/prcn_prcn_seed42.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model = PRCN(bio_dim=len(grp['biophysical']), econ_dim=len(grp['economic']),
                 hist_dim=len(grp['historical']), cascade_dim=len(grp['cascade']))
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # Build batch from county's monthly data
    def norm_cols(df, cols):
        X = df[cols].values.astype(np.float32)
        for i, c in enumerate(cols):
            if c in stats: X[:, i] = (X[:, i] - stats[c]['mean']) / max(stats[c]['std'], 1e-8)
        return torch.from_numpy(np.nan_to_num(X))

    batch = {
        'x_bio': norm_cols(county_data, grp['biophysical']),
        'x_econ': norm_cols(county_data, grp['economic']),
        'x_hist': norm_cols(county_data, grp['historical']),
        'x_cascade': torch.from_numpy(county_data[grp['cascade']].values.astype(np.float32)),
    }
    with torch.no_grad():
        out = model(batch)

    months = county_data['month'].values
    cw = out['channel_weights'].numpy()  # (T, 12)
    vuln = out['persistence_score'].numpy().flatten()
    waste_prob = torch.sigmoid(out['waste_logits']).numpy().flatten()
    actual_waste = labs[mask].sort_values('month')['waste'].values

    # Plot
    channel_names = grp['cascade']
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    # Panel A: channel routing weights heatmap
    im = axes[0].imshow(cw.T, aspect='auto', cmap='Reds', extent=[months[0]-0.5, months[-1]+0.5, -0.5, len(channel_names)-0.5])
    axes[0].set_yticks(range(len(channel_names)))
    axes[0].set_yticklabels([c.replace('_', ' ') for c in channel_names], fontsize=6)
    axes[0].set_title(f'Vulnerability-Conditioned Channel Routing — FIPS {target_fips} ({target_comm}) 2023', fontweight='bold')
    plt.colorbar(im, ax=axes[0], label='Activation', shrink=0.8)

    # Panel B: persistence score + vulnerability
    axes[1].plot(months, vuln, 'o-', color='#e74c3c', label='Cascade persistence', linewidth=2)
    axes[1].set_ylabel('Score'); axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

    # Panel C: predicted vs actual waste
    axes[2].bar(months - 0.15, actual_waste, 0.3, color='#95a5a6', label='Actual waste', alpha=0.7)
    axes[2].bar(months + 0.15, waste_prob, 0.3, color='#e74c3c', label='Predicted P(waste)', alpha=0.7)
    axes[2].set_xlabel('Month'); axes[2].set_ylabel('Waste'); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.3)
    axes[2].set_xticks(months)

    plt.tight_layout()
    fig.savefig('paper/figures/fig4_case_study.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig4_case_study.pdf')
else:
    print(f'No checkpoint at {ckpt_path} — run training first')

---
## Download

In [ ]:
backup()
OUT = f'{WORKDIR}/cascrop_prcn_results.tar.gz'
!tar czf {OUT} results/ paper/figures/ paper/tables/ checkpoints/
if ON_COLAB:
    try:
        from google.colab import files; files.download(OUT)
    except: print(f'Download from Files panel: {OUT}')
elif ON_KAGGLE:
    print(f'Results saved to {OUT} — find in Output tab')
else:
    print(f'Results at {OUT}')
print('DONE')